# 第四周练习 —— 代码文档助手（Code Documentation Assistant）

## 练习目标

构建一个小工具：粘贴源代码 → 选模型 → 让 LLM **自动识别语言并补文档注释**（docstring / JSDoc 等），原样返回完整代码。

## 和本课第 4 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| System / User Prompt | `system_prompt` 定规则；`user_prompt(code)` 塞入待注释代码 |
| 多模型路由 | OpenRouter 统一入口，下拉切换 GPT / Claude / Gemini |
| Gradio Blocks | 左右分栏：源码输入 vs 文档化输出 |

## 怎么跑

1. `.env` 里准备 `OPENROUTER_API_KEY`（可选再配 `GROQ_API_KEY`）
2. 从上到下依次运行单元格
3. 在 Gradio 里粘贴代码、选模型，点 **Generate Documentation**


In [ ]:
# 导入 os：读环境变量（API Key）
import os
# 导入 OpenAI 官方 SDK：后面用它走 OpenRouter 兼容接口
from openai import OpenAI
# 导入 Gradio：搭简易 Web UI
import gradio as gr
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进进程环境
from dotenv import load_dotenv


In [ ]:
# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)
# 从环境读取 OpenRouter API Key（发给 OpenRouter 网关）
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
# 顺带读取 Groq Key（本笔记本后续未直接用到，保留原逻辑）
groq_api_key = os.getenv("GROQ_API_KEY")

# 有 key 就提示已设置，方便自检（勿打印完整密钥）
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    # 未设置时提醒：后面调用会失败
    print("OPENROUTER_API_KEY is not set.")


In [ ]:
# ========== 模型与网关常量（OpenRouter 路由名，勿改字符串） ==========
# GPT 侧：OpenRouter 上的 openai/gpt-4o-mini
MODEL_GPT = 'openai/gpt-4o-mini'
# Claude 侧：Haiku 轻量档
MODEL_CLAUDE = 'anthropic/claude-haiku-4.5'
# Gemini 侧：Flash 快速档
MODEL_GEMINI = 'google/gemini-2.5-flash'
# OpenRouter 的 OpenAI 兼容 Base URL
OPENROUTER_URL = "https://openrouter.ai/api/v1"


In [ ]:
# 创建 OpenAI 客户端，但把 base_url 指到 OpenRouter：一套 SDK 调多家模型
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_URL)


In [ ]:
# system prompt：规定「识别语言 + 按该语言惯例写文档注释」，整段发给模型，勿译勿改
system_prompt = """You are a code documentation assistant.

Your task is to analyze provided source code, detect the programming language, and add clear, idiomatic documentation comments appropriate for that language.

Rules

- Automatically detect the programming language.

- Use the correct documentation style for that language (e.g., docstrings, JSDoc, Javadoc, GoDoc, Doxygen, etc.).

- Preserve original code exactly — do not refactor, rename, or restructure.

- Focus on intent and behavior, not trivial syntax.

- Keep comments concise and professional.

- Do not over-comment obvious lines.

Output Requirements

- Return the complete documented code.

- Output only a single fenced code block labeled with the detected language.

- Do not include explanations outside the code block.

If the language is ambiguous, respond with failed to detect language and do not return any code.
"""


In [ ]:
# 根据用户粘贴的源码拼 user 消息：任务说明 + 代码正文
def user_prompt(code):
    # f-string 把 code 嵌进提示；英文指令保持原样（影响模型行为）
    return f"""Analyze the following source code and add appropriate documentation comments:
{code}
"""


In [ ]:
# 核心：调用 OpenRouter，用选定 model 给代码加文档注释
def generate_comments(code,model):
    # chat.completions.create：标准 Chat Completions；model 来自 Gradio 下拉
    response = openrouter.chat.completions.create(
        model=model,
        messages=[
            # system：文档化规则
            {"role": "system", "content": system_prompt},
            # user：具体源码
            {"role": "user", "content": user_prompt(code)}
        ],
        # 限制最大生成长度，避免超长回复
        max_tokens=2048,
        # 较低 temperature：注释更稳、少发挥
        temperature=0.2
    )
    # 取第一条 choice 的文本内容返回给 Gradio
    return response.choices[0].message.content


In [ ]:
# 用 Gradio Blocks 搭「左源码 / 右文档化结果」界面
with gr.Blocks() as demo:
    # 标题 Markdown（UI 文案保持原英文）
    gr.Markdown("# Code Documentation Assistant")
    # 第一行：两列 —— 输入源码 vs 输出带注释代码
    with gr.Row():
        with gr.Column():
            # Code 组件：适合编辑多行源码
            code_input = gr.Code(label="Source Code")
        with gr.Column():
            # 右侧展示模型返回的文档化代码
            output = gr.Code(label="Documented Code")
    # 第二行：模型选择 + 生成按钮
    with gr.Row():
        with gr.Column():
            # Dropdown：在 GPT / Claude / Gemini 三个 OpenRouter 路由名之间切换
            model_selector = gr.Dropdown(label="Select Model", choices=[MODEL_GPT, MODEL_CLAUDE, MODEL_GEMINI], value=MODEL_GPT, interactive=True)
        with gr.Column():
            # 主按钮：触发 generate_comments
            doc_button = gr.Button("Generate Documentation", variant="primary")
    # 点击：把 code_input + model_selector 传给 generate_comments，结果写入 output
    doc_button.click(fn=generate_comments, inputs=[code_input, model_selector], outputs=output)
# inbrowser=True：启动后尝试在浏览器打开
demo.launch(inbrowser=True)
